# NovaCart — Silver Geolocation Transformation

## 1. Import Libraries

In [0]:
from pyspark.sql import functions as F

## 2. Define Storage Paths

In [0]:
BRONZE_GEOLOCATION_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/geolocation"
)

SILVER_GEOLOCATION_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/geolocation"
)

QUARANTINE_GEOLOCATION_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/geolocation"
)

print(f"Bronze path: {BRONZE_GEOLOCATION_PATH}")
print(f"Silver path: {SILVER_GEOLOCATION_PATH}")
print(f"Quarantine path: {QUARANTINE_GEOLOCATION_PATH}")

## 3. Read Bronze Geolocation Data

In [0]:
geolocation_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_GEOLOCATION_PATH)
)

bronze_row_count = geolocation_bronze_df.count()

print("Bronze geolocation loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

geolocation_bronze_df.printSchema()
display(geolocation_bronze_df.limit(10))

## 4. Validate Required Columns

In [0]:
required_columns = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in geolocation_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 5. Profile Missing and Invalid Geolocation Values

In [0]:
geolocation_profile_df = geolocation_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("geolocation_zip_code_prefix").isNull()
            | (F.col("geolocation_zip_code_prefix") < 0)
            | (F.col("geolocation_zip_code_prefix") > 99999)
        ).cast("int")
    ).alias("invalid_zip_code_prefix"),

    F.sum(
        (
            F.col("geolocation_lat").isNull()
            | (F.col("geolocation_lat") < -90)
            | (F.col("geolocation_lat") > 90)
        ).cast("int")
    ).alias("invalid_latitude"),

    F.sum(
        (
            F.col("geolocation_lng").isNull()
            | (F.col("geolocation_lng") < -180)
            | (F.col("geolocation_lng") > 180)
        ).cast("int")
    ).alias("invalid_longitude"),

    F.sum(
        (
            F.col("geolocation_city").isNull()
            | (F.trim(F.col("geolocation_city")) == "")
        ).cast("int")
    ).alias("invalid_city"),

    F.sum(
        (
            F.col("geolocation_state").isNull()
            | (F.trim(F.col("geolocation_state")) == "")
        ).cast("int")
    ).alias("invalid_state"),

    F.sum(
        (
            F.col("geolocation_state").isNotNull()
            & ~F.trim(F.col("geolocation_state")).rlike("^[A-Za-z]{2}$")
        ).cast("int")
    ).alias("invalid_state_format"),
)

display(geolocation_profile_df)

## 6. Print Full Geolocation Quality Profile

In [0]:
profile = geolocation_profile_df.first().asDict()

for metric, value in profile.items():
    print(f"{metric}: {value}")

## 7. Check Exact Duplicate Geolocation Records

In [0]:
business_columns = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
]

exact_duplicate_count = (
    bronze_row_count
    - geolocation_bronze_df
        .dropDuplicates(business_columns)
        .count()
)

print(f"Exact duplicate geolocation rows: {exact_duplicate_count}")

## 8. Profile ZIP Code Repetition

In [0]:
zip_profile_df = (
    geolocation_bronze_df
    .groupBy("geolocation_zip_code_prefix")
    .count()
    .orderBy(F.desc("count"))
)

distinct_zip_count = zip_profile_df.count()

print(f"Distinct ZIP code prefixes: {distinct_zip_count}")

display(zip_profile_df.limit(20))

## 9. Clean and Standardize Geolocation Fields

In [0]:
geolocation_cleaned_df = (
    geolocation_bronze_df
    .withColumn(
        "geolocation_city",
        F.lower(F.trim(F.col("geolocation_city")))
    )
    .withColumn(
        "geolocation_state",
        F.upper(F.trim(F.col("geolocation_state")))
    )
)

## 10. Define Geolocation Validation Rules

In [0]:
invalid_zip_code_condition = (
    F.col("geolocation_zip_code_prefix").isNull()
    | (F.col("geolocation_zip_code_prefix") < 0)
    | (F.col("geolocation_zip_code_prefix") > 99999)
)

invalid_latitude_condition = (
    F.col("geolocation_lat").isNull()
    | (F.col("geolocation_lat") < -90)
    | (F.col("geolocation_lat") > 90)
)

invalid_longitude_condition = (
    F.col("geolocation_lng").isNull()
    | (F.col("geolocation_lng") < -180)
    | (F.col("geolocation_lng") > 180)
)

invalid_city_condition = (
    F.col("geolocation_city").isNull()
    | (F.col("geolocation_city") == "")
)

invalid_state_condition = (
    F.col("geolocation_state").isNull()
    | (F.col("geolocation_state") == "")
    | ~F.col("geolocation_state").rlike("^[A-Z]{2}$")
)

## 11. Assign Geolocation Rejection Reasons

In [0]:
geolocation_validated_df = geolocation_cleaned_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_zip_code_condition,
        F.lit("INVALID_ZIP_CODE_PREFIX")
    )
    .when(
        invalid_latitude_condition,
        F.lit("INVALID_LATITUDE")
    )
    .when(
        invalid_longitude_condition,
        F.lit("INVALID_LONGITUDE")
    )
    .when(
        invalid_city_condition,
        F.lit("MISSING_CITY")
    )
    .when(
        invalid_state_condition,
        F.lit("INVALID_STATE")
    )
    .otherwise(F.lit(None))
)

## 12. Review Geolocation Validation Results

In [0]:
display(
    geolocation_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 13. Split Valid and Invalid Geolocation Records

In [0]:
geolocation_valid_df = (
    geolocation_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop("_rejection_reason")
)

geolocation_quarantine_df = (
    geolocation_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
)

## 14. Add Silver Processing Metadata

In [0]:
geolocation_silver_df = (
    geolocation_valid_df
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 15. Add Quarantine Metadata

In [0]:
geolocation_quarantine_df = (
    geolocation_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("geolocation")
    )
)

## 16. Count Silver and Quarantine Records

In [0]:
valid_row_count = geolocation_silver_df.count()
quarantine_row_count = geolocation_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 17. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 18. Write Valid Geolocation Records to Silver

In [0]:
(
    geolocation_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_GEOLOCATION_PATH)
)

print("Silver geolocation written successfully.")

## 19. Write Invalid Geolocation Records to Quarantine

In [0]:
(
    geolocation_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_GEOLOCATION_PATH)
)

print("Geolocation quarantine output written successfully.")

## 20. Read Written Delta Outputs

In [0]:
geolocation_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_GEOLOCATION_PATH)
)

geolocation_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_GEOLOCATION_PATH)
)

silver_written_count = geolocation_silver_written_df.count()
quarantine_written_count = geolocation_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 21. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver geolocation pipeline completed successfully.")
print("Final row-count validation passed.")

## 22. Inspect Final Silver Geolocation Dataset

In [0]:
geolocation_silver_written_df.printSchema()

display(
    geolocation_silver_written_df.select(
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state",
        "_silver_processed_at"
    ).limit(20)
)